In [ ]:
#| default_exp tree

In [ ]:
#| hide
from nbdev.showdoc import *

# tree
> Parse a PDF into a hierarchy tree, derive a vectorless knowledge graph (hierarchy + cross-reference + entity edges), and use it to enrich search results (TreeRAG).

`litesearch` normally ingests PDFs as a flat list of chunks. This module adds *structural* retrieval: it builds a per-document graph — chapter/section hierarchy plus algorithmically-derived cross-reference and entity-overlap edges — so a search hit can be enriched by walking the graph (parent context, siblings, referenced sections) rather than relying on vector similarity alone.

Ingestion is an **auto-fallback hybrid**: `pdf_oxide` extracts headings first, and if its heading detection looks poor (e.g. a scanned/OCR page) *and* the optional `docling` extra is installed, docling is used for heading structure instead. Chunking and embedding always stay on `pdf_oxide`.

In [ ]:
#| export
from fastcore.all import L, first, Path
from pdf_oxide import PdfDocument
from dataclasses import dataclass, field
from collections import defaultdict
import re, json, statistics

## Data structures & schema

A `DocNode` is one node of a document hierarchy. The graph is persisted in two tables:

- `doc_tree` — one row per node (title, level, page, parent, section number).
- `doc_edges` — one row per edge: `etype` is `parent`, `xref`, or `entity`.

Chunk text/embeddings live in the usual `store` table; each store row's `metadata` JSON carries `doc` and `node_id` so a hit can be mapped back to its tree node.

In [ ]:
#| export
@dataclass
class DocNode:
    "A node in a document hierarchy tree."
    id: int                       # per-document node id (0 = synthetic root)
    doc: str                      # document name
    title: str                    # heading text
    level: int                    # 0 = root, 1 = top-level section, ...
    page: int = 0                 # 0-indexed page the heading appears on
    parent_id: int | None = None  # id of the parent node
    num: str | None = None        # section number, e.g. "3.1.2"
    children: list = field(default_factory=list)   # child node ids
    chunk_ids: list = field(default_factory=list)   # global chunk indices

def tree_tables(db,                  # litesearch database
                tree='doc_tree',     # hierarchy table name
                edges='doc_edges',   # edge table name
                ):
    "Create the `doc_tree` and `doc_edges` tables if absent; returns (tree_table, edges_table)."
    t = db.t[tree].create(id=int, doc=str, node_id=int, title=str, level=int, page=int,
                          parent_id=int, num=str, store_id=int, pk='id', if_not_exists=True)
    e = db.t[edges].create(id=int, doc=str, src=int, dst=int, etype=str, weight=float,
                           label=str, pk='id', if_not_exists=True)
    return t, e

## Heading extraction

Three independent producers, each returning an `L` of `dict(title, level, page, num, source)`:

- `outline_headings` — the embedded PDF outline (often title-less on academic PDFs).
- `span_headings` — line-level font analysis: a line is a heading if it starts with a section number *and* is bold, or is bold and in a heading-sized font band.
- `md_headings` — `#`-prefixed lines from `pdf_oxide`'s markdown conversion.

`extract_headings` unions and de-duplicates them, preferring entries that carry a section number.

In [ ]:
#| export
def _page_lines(doc,   # PdfDocument
                pg,    # 0-indexed page number
                ):
    "Group a page into lines with aggregated font metadata: dict(text, page, y, font_size, bold_frac)."
    out = []
    for ln in doc.extract_text_lines(pg):
        text = re.sub(r'\s+', ' ', ln.text or '').strip()
        if not text: continue
        words = [w for w in (ln.words or []) if (w.text or '').strip()]
        if not words: continue
        bold = sum(bool(getattr(w, 'is_bold', False)) for w in words) / len(words)
        size = max((getattr(w, 'font_size', 0) or 0) for w in words)
        y = ln.bbox[1] if getattr(ln, 'bbox', None) else 0
        out.append(dict(text=text, page=pg, y=y, font_size=size, bold_frac=bold))
    return sorted(out, key=lambda r: -r['y'])

_NUM_RE = re.compile(r'^(\d+(?:\.\d+)*)\.?\s+(\S.*)$')

def span_headings(doc,            # PdfDocument
                  max_words=12,   # ignore lines longer than this (body text)
                  ):
    "Detect headings from section-number and font cues; returns L of heading dicts."
    out = L()
    for pg in range(doc.page_count()):
        lines = _page_lines(doc, pg)
        sizes = [ln['font_size'] for ln in lines if ln['font_size']]
        med = statistics.median(sizes) if sizes else 0
        for ln in lines:
            txt, sz, bf = ln['text'], ln['font_size'], ln['bold_frac']
            if len(txt) > 100 or len(txt.split()) > max_words or len(txt) < 3: continue
            m = _NUM_RE.match(txt)
            big = bool(med) and med * 1.15 < sz < med * 2.0
            if m and bf > 0.6 and re.search(r'[A-Za-z]', m.group(2)):
                num = m.group(1)
                out.append(dict(title=txt, level=num.count('.') + 1, page=pg,
                                num=num, source='span', y=ln['y']))
            elif bf > 0.8 and big and not txt.endswith('.') and re.search(r'[A-Za-z]', txt):
                out.append(dict(title=txt, level=1, page=pg, num=None,
                                source='span', y=ln['y']))
    return out

In [ ]:
#| export
def outline_headings(doc,   # PdfDocument
                     ):
    "Headings from the embedded PDF outline; empty L if all titles are blank."
    out = L()
    def walk(items, level):
        for it in (items or []):
            t = (it.get('title') or '').strip()
            pg = max(0, (it.get('page') or 1) - 1)   # outline pages are 1-indexed
            if t and t != '(No Title)':
                out.append(dict(title=t, level=level, page=pg, num=None,
                                source='outline', y=0))
            walk(it.get('children'), level + 1)
    walk(doc.get_outline(), 1)
    return out

_MD_RE = re.compile(r'^(#+)\s+(.+)$')

def md_headings(doc,   # PdfDocument
                ):
    "Headings from `#`-prefixed lines of pdf_oxide's markdown conversion."
    out = L()
    for pg in range(doc.page_count()):
        for line in doc.to_markdown(pg, detect_headings=True).splitlines():
            m = _MD_RE.match(line.strip())
            if not m: continue
            title = m.group(2).strip()
            if not title or re.search(r'arxiv', title, re.I): continue
            out.append(dict(title=title, level=len(m.group(1)), page=pg,
                            num=None, source='md', y=0))
    return out

In [ ]:
#| export
def _hkey(h):
    "De-dup key for a heading: normalised title + page."
    return (re.sub(r'\W+', '', h['title'].lower())[:40], h['page'])

def extract_headings(doc,   # PdfDocument
                     ):
    "Union span/outline/markdown headings, de-duplicate, and return them in document order."
    cand = span_headings(doc) + outline_headings(doc) + md_headings(doc)
    idx, out = {}, []
    for h in cand:
        k = _hkey(h)
        if not k[0]: continue
        if k in idx:
            ex = out[idx[k]]
            if not ex.get('num') and h.get('num'): out[idx[k]] = h
            continue
        idx[k] = len(out); out.append(h)
    out.sort(key=lambda h: (h['page'], -h.get('y', 0)))
    return L(out)

In [ ]:
PDF = first(p for p in [Path('pdfs/attention_is_all_you_need.pdf'),
                        Path('nbs/pdfs/attention_is_all_you_need.pdf')] if p.exists())
_doc = PdfDocument(str(PDF))
_h = extract_headings(_doc)
assert len(_h) >= 10, f'too few headings: {len(_h)}'
assert any(x.get('num') == '3.1' for x in _h), 'numbered heading 3.1 not recovered'
assert any('Introduction' in x['title'] for x in _h), 'Introduction heading missing'
print(f'{len(_h)} headings; first 6:', [x['title'] for x in _h[:6]])

## Tree construction & chunk assignment

`build_doc_tree` nests headings under a synthetic level-0 root using a stack: a heading's parent is the nearest preceding heading of a lower level. `assign_chunks` then attaches every chunk to the last node whose page is at or before the chunk's page (chunks before the first heading land on the root).

In [ ]:
#| export
def build_doc_tree(headings,   # ordered L of heading dicts
                   doc_name,   # document name (used for the root title)
                   ):
    "Build a hierarchy: a synthetic level-0 root with headings nested by level."
    root = DocNode(id=0, doc=doc_name, title=doc_name, level=0, page=0, parent_id=None)
    nodes, stack = [root], [root]
    for h in headings:
        node = DocNode(id=len(nodes), doc=doc_name, title=h['title'],
                       level=max(1, h['level']), page=h['page'], num=h.get('num'))
        while stack and stack[-1].level >= node.level: stack.pop()
        parent = stack[-1] if stack else root
        node.parent_id = parent.id
        parent.children.append(node.id)
        nodes.append(node)
        stack.append(node)
    return nodes

def assign_chunks(nodes,    # list of DocNode (from build_doc_tree)
                  chunks,   # ordered iterable of (page, chunk_idx, text)
                  ):
    "Attach each chunk (by global index) to the last node whose page <= the chunk's page."
    ordered = sorted([n for n in nodes if n.level > 0], key=lambda n: (n.page, n.id))
    root = nodes[0]
    for gi, (pg, ci, text) in enumerate(chunks):
        target = root
        for n in ordered:
            if n.page <= pg: target = n
            else: break
        target.chunk_ids.append(gi)
    return nodes

In [ ]:
_nodes = build_doc_tree(_h, PDF.name)
_root = _nodes[0]
assert _root.level == 0 and _root.parent_id is None, 'bad root'
assert _root.children, 'root has no children'
assert all(n.parent_id is not None for n in _nodes if n.level > 0), 'orphan node'
assert all(0 <= n.parent_id < len(_nodes) for n in _nodes if n.parent_id is not None), 'bad parent id'
print(f'{len(_nodes)} nodes, max depth {max(n.level for n in _nodes)}')

## Graph edges

Two algorithmic (vectorless) edge types on top of the `parent` hierarchy:

- `xref_edges` — regex over chunk text: `Section X.Y` references resolve to the node with that number; `Figure/Table N` references resolve to the node that owns the caption; bracketed citations link to a `References` node.
- `entity_edges` — YAKE keywords per node; non-parent/child node pairs sharing enough keywords get a co-occurrence edge.

In [ ]:
#| export
_SEC_RE = re.compile(r'\b(?:Section|Sec)\.?\s+(\d+(?:\.\d+)*)', re.I)
_FIG_RE = re.compile(r'\b(Figure|Fig|Table)\.?\s+(\d+)', re.I)
_CAP_RE = re.compile(r'\b(Figure|Table)\s+(\d+)\s*[:.]', re.I)
_CITE_RE = re.compile(r'\[\d+(?:\s*,\s*\d+)*\]')

def xref_edges(nodes,    # list of DocNode (with chunk_ids assigned)
               chunks,   # ordered iterable of (page, chunk_idx, text)
               ):
    "Cross-reference edges from Section/Figure/Table references and citations in chunk text."
    chunks = list(chunks)
    by_num = {n.num: n.id for n in nodes if n.num}
    gi2node = {gi: n.id for n in nodes for gi in n.chunk_ids}
    refs_node = first((n.id for n in nodes if re.match(r'\s*references\b', n.title, re.I)))
    owns = {}
    for gi, (_, _, text) in enumerate(chunks):
        nid = gi2node.get(gi)
        if nid is None: continue
        for m in _CAP_RE.finditer(text):
            owns.setdefault((m.group(1).lower(), m.group(2)), nid)
    edges, seen = L(), set()
    def add(src, dst, label):
        if src is None or dst is None or src == dst: return
        k = (src, dst, label)
        if k in seen: return
        seen.add(k)
        edges.append(dict(src=src, dst=dst, etype='xref', weight=1.0, label=label))
    for gi, (_, _, text) in enumerate(chunks):
        src = gi2node.get(gi)
        if src is None: continue
        for m in _SEC_RE.finditer(text):
            add(src, by_num.get(m.group(1)), f'Section {m.group(1)}')
        for m in _FIG_RE.finditer(text):
            kind = 'figure' if m.group(1).lower().startswith('fig') else m.group(1).lower()
            add(src, owns.get((kind, m.group(2))), f'{m.group(1)} {m.group(2)}')
        if refs_node is not None and _CITE_RE.search(text):
            add(src, refs_node, 'citation')
    return edges

In [ ]:
#| export
def entity_edges(nodes,         # list of DocNode
                 node_text,     # dict node_id -> concatenated chunk text
                 top_k=8,       # keywords extracted per node
                 min_shared=2,  # min shared keywords for an edge
                 ):
    "Co-occurrence edges between nodes that share >= min_shared YAKE keywords."
    try:
        from yake import KeywordExtractor
    except ImportError:
        return L()
    kw = KeywordExtractor(top=top_k)
    keys = {}
    for n in nodes:
        txt = (node_text.get(n.id) or '').strip()
        if len(txt) < 40: continue
        keys[n.id] = {k.lower() for k, _ in kw.extract_keywords(txt)}
    pairs = {(n.parent_id, n.id) for n in nodes} | {(n.id, n.parent_id) for n in nodes}
    edges, ids = L(), list(keys)
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            a, b = ids[i], ids[j]
            if (a, b) in pairs: continue
            shared = keys[a] & keys[b]
            if len(shared) >= min_shared:
                edges.append(dict(src=a, dst=b, etype='entity',
                                  weight=round(len(shared) / top_k, 3),
                                  label=', '.join(sorted(shared)[:3])))
    return edges

In [ ]:
# Hub-free paragraph chunker so tests run offline (the default pdf_chunks() needs
# a chonkie recipe download); passed to ingest via the encode_pdf_texts `chunk_fn` hook.
def _para_chunks(doc, min_len=80):
    "Paragraph-split pdf_oxide page text - test scaffold, no network."
    out = []
    for pg in range(doc.page_count()):
        for ci, para in enumerate(re.split(r'\n\s*\n', doc.extract_text(pg) or '')):
            para = re.sub(r'\s+', ' ', para).strip()
            if len(para) >= min_len: out.append((pg, ci, para))
    return L(out)

_chunks = _para_chunks(_doc)
assign_chunks(_nodes, _chunks)
assert sum(len(n.chunk_ids) for n in _nodes) == len(_chunks), 'chunk count mismatch'
assert all(gi in range(len(_chunks)) for n in _nodes for gi in n.chunk_ids), 'bad chunk id'

_nt = defaultdict(str)
for n in _nodes:
    for gi in n.chunk_ids: _nt[n.id] += _chunks[gi][2] + '\n'
_xe = xref_edges(_nodes, _chunks)
_ee = entity_edges(_nodes, dict(_nt))
assert len(_xe) >= 1, 'no cross-reference edges found'
assert len(_ee) >= 1, 'no entity edges found'
print(f'{len(_xe)} xref edges, {len(_ee)} entity edges; xref sample:', _xe[0]['label'])

## Ingest — auto-fallback hybrid

`ingest_pdf_tree` runs `pdf_oxide` heading extraction first and scores it with `_heading_quality`. When `backend='auto'` and the score is below `quality_thresh`, it tries `docling_headings` (optional `docling` extra) for better structure on scanned/OCR pages; if `docling` is absent it warns once and keeps the `pdf_oxide` headings. Chunking and embedding always use `pdf_oxide`.

In [ ]:
#| export
def _heading_quality(headings,   # L of heading dicts
                     doc,        # PdfDocument
                     ):
    "Score pdf_oxide heading output in 0..1; low scores trigger the docling fallback."
    n = len(headings)
    if n == 0: return 0.0
    pc = max(1, doc.page_count())
    numbered = sum(1 for h in headings if h.get('num'))
    pages = {h['page'] for h in headings}
    count_score = min(1.0, n / max(3, pc / 4))
    num_score = numbered / n
    cover_score = min(1.0, len(pages) / max(1, pc / 3))
    return round(0.4 * count_score + 0.35 * num_score + 0.25 * cover_score, 3)

def docling_headings(path,   # path to a PDF
                     ):
    "Section headings via the optional `docling` extra, mapped to the standard heading shape."
    try:
        from docling.document_converter import DocumentConverter
    except ImportError as e:
        raise ImportError("docling is not installed - install the extra: "
                          "pip install 'litesearch[docling]'") from e
    dl = DocumentConverter().convert(str(path)).document
    out = L()
    for it in getattr(dl, 'texts', []):
        label = str(getattr(it, 'label', '')).lower()
        is_title = 'title' in label
        if 'section_header' not in label and not is_title: continue
        text = (getattr(it, 'text', '') or '').strip()
        if not text: continue
        prov = getattr(it, 'prov', None) or []
        page = max(0, getattr(prov[0], 'page_no', 1) - 1) if prov else 0
        lvl = getattr(it, 'level', None)
        level = 1 if is_title else max(1, int(lvl) if lvl else 1)
        m = _NUM_RE.match(text)
        out.append(dict(title=text, level=level, page=page,
                        num=(m.group(1) if m else None), source='docling', y=0))
    return out

In [ ]:
#| export
def ingest_pdf_tree(db,                   # litesearch database
                    path,                 # path to a PDF
                    enc,                  # FastEncode-style text encoder
                    store='store',        # store table name
                    backend='auto',       # 'auto' | 'oxide' | 'docling'
                    quality_thresh=0.4,   # auto-fallback heading-quality threshold
                    tree='doc_tree',      # hierarchy table name
                    edges='doc_edges',    # edge table name
                    **kw,                 # forwarded to encode_pdf_texts / pdf_chunks
                    ):
    "Ingest a PDF as a hierarchy tree + knowledge graph; embeddings go to the store table."
    import litesearch.data  # noqa: F401 - applies PdfDocument chunking patches
    from litesearch.utils import encode_pdf_texts
    doc = PdfDocument(str(path))
    doc_name = Path(path).name
    headings = extract_headings(doc)
    used = 'oxide'
    if backend == 'docling' or (backend == 'auto'
                                and _heading_quality(headings, doc) < quality_thresh):
        try:
            dh = docling_headings(path)
            if dh: headings, used = dh, 'docling'
        except ImportError as e:
            if backend == 'docling': raise
            import warnings
            warnings.warn(f'{e}; keeping pdf_oxide headings')
    nodes = build_doc_tree(headings, doc_name)
    embedded = encode_pdf_texts(doc, enc, **kw)            # L of (page, ci, text, emb)
    chunks = L((pg, ci, text) for pg, ci, text, _ in embedded)
    assign_chunks(nodes, chunks)

    store_tbl = db.get_store(store)
    tt, et = tree_tables(db, tree, edges)
    gi2node = {gi: n.id for n in nodes for gi in n.chunk_ids}
    node_text, rowids = defaultdict(str), []
    for gi, (pg, ci, text, emb) in enumerate(embedded):
        nid = gi2node.get(gi, 0)
        node_text[nid] += text + '\n'
        meta = json.dumps(dict(doc=doc_name, node_id=nid, page=pg, backend=used))
        rec = store_tbl.insert(dict(content=text, embedding=emb.tobytes(), metadata=meta))
        rowids.append(rec['id'])
    for n in nodes:
        sid = rowids[n.chunk_ids[0]] if n.chunk_ids else None
        tt.insert(dict(doc=doc_name, node_id=n.id, title=n.title, level=n.level,
                       page=n.page, parent_id=n.parent_id, num=n.num, store_id=sid))
    pe = L(dict(src=n.id, dst=n.parent_id, etype='parent', weight=1.0, label='parent')
           for n in nodes if n.parent_id is not None)
    all_edges = pe + xref_edges(nodes, chunks) + entity_edges(nodes, dict(node_text))
    for e in all_edges:
        et.insert(dict(doc=doc_name, **e))
    return dict(doc=doc_name, backend=used, n_nodes=len(nodes),
                n_chunks=len(embedded), n_edges=len(all_edges), nodes=nodes)

## Retrieval — TreeRAG

`tree_expand_results` is a standalone primitive: given search hits, it maps each hit to its tree node and walks `doc_edges` up to `hops` steps, attaching neighbour store rows, a heading `breadcrumb`, and a short `parent_intro`. `tree_search` wraps `db.search` with this expansion and re-ranks (graph neighbours are demoted by `alpha * graph_distance`). `build_tree_rag_prompt` assembles the enriched context into a prompt.

In [ ]:
#| export
def _breadcrumb(db, doc, node_id, tree='doc_tree'):
    "Heading path from the root to `node_id`, joined with ' > '."
    titles, cur, guard = [], node_id, 0
    while cur is not None and guard < 64:
        rows = list(db.query(f'SELECT title, parent_id FROM {tree} '
                             f'WHERE doc=:d AND node_id=:n', dict(d=doc, n=cur)))
        if not rows: break
        titles.append(rows[0]['title'])
        cur = rows[0]['parent_id']
        guard += 1
    return ' > '.join(reversed(titles))

def _parent_intro(db, doc, node_id, tree='doc_tree', store='store', maxlen=240):
    "First chunk of the parent node, truncated - a short section-context snippet."
    rows = list(db.query(f'SELECT parent_id FROM {tree} WHERE doc=:d AND node_id=:n',
                         dict(d=doc, n=node_id)))
    if not rows or rows[0]['parent_id'] is None: return ''
    crows = list(db.query(f"SELECT content FROM {store} "
                          f"WHERE json_extract(metadata,'$.doc')=:d "
                          f"AND json_extract(metadata,'$.node_id')=:n LIMIT 1",
                          dict(d=doc, n=rows[0]['parent_id'])))
    return (crows[0]['content'][:maxlen] if crows else '')

def tree_expand_results(db,                                      # litesearch database
                        results,                                 # rows from db.search
                        hops=1,                                  # graph hops to expand
                        etypes=('parent', 'xref', 'entity'),     # edge types to follow
                        tree='doc_tree', edges='doc_edges', store='store',
                        per_node=2,                              # neighbour rows per node
                        ):
    "Enrich search hits with document-graph context (breadcrumb, parent intro, neighbours)."
    results = list(results or [])
    if not results: return results
    q = lambda sql, p: list(db.query(sql, p))
    seen = {r.get('rowid') for r in results}
    out = []
    for r in results:
        meta = r.get('metadata')
        try: m = json.loads(meta) if isinstance(meta, str) else (meta or {})
        except Exception: m = {}
        doc, nid = m.get('doc'), m.get('node_id')
        row = dict(r, _graph_dist=0)
        if doc is not None and nid is not None:
            row['breadcrumb'] = _breadcrumb(db, doc, nid, tree)
            row['parent_intro'] = _parent_intro(db, doc, nid, tree, store)
        out.append(row)
        if doc is None or nid is None: continue
        visited, frontier = {nid}, {nid}
        for d in range(1, hops + 1):
            nxt = set()
            for s in frontier:
                for e in q(f'SELECT src, dst, etype FROM {edges} '
                           f'WHERE doc=:d AND (src=:s OR dst=:s)', dict(d=doc, s=s)):
                    if e['etype'] not in etypes: continue
                    nb = e['dst'] if e['src'] == s else e['src']
                    if nb in visited: continue
                    visited.add(nb); nxt.add(nb)
            frontier = nxt
            for nb in nxt:
                for srow in q(f"SELECT id AS rowid, content, metadata FROM {store} "
                              f"WHERE json_extract(metadata,'$.doc')=:d "
                              f"AND json_extract(metadata,'$.node_id')=:n LIMIT :lim",
                              dict(d=doc, n=nb, lim=per_node)):
                    if srow['rowid'] in seen: continue
                    seen.add(srow['rowid'])
                    out.append(dict(srow, _graph_dist=d,
                                    breadcrumb=_breadcrumb(db, doc, nb, tree)))
    return out

In [ ]:
#| export
def tree_search(db,                  # litesearch database
                q,                   # query string
                enc,                 # FastEncode-style encoder
                expand=True,         # enrich hits with graph context
                hops=1,              # graph hops to expand
                alpha=0.15,          # per-hop score penalty
                columns=None,        # store columns to fetch
                table_name='store',  # store table name
                limit=50,            # max base hits
                **kw,                # forwarded to db.search
                ):
    "Search the store, then enrich and re-rank hits with document-graph context."
    if not q or not q.strip(): return []
    cols = list(columns or ['content', 'metadata'])
    emb = enc.encode_query([q])[0].tobytes()
    results = list(db.search(q, emb, columns=cols, table_name=table_name,
                             limit=limit, **kw) or [])
    if not expand or not results: return results
    enriched = tree_expand_results(db, results, hops=hops, store=table_name)
    for r in enriched:
        r['_tree_score'] = r.get('_rrf_score', 0.0) - alpha * r.get('_graph_dist', 0)
    return sorted(enriched, key=lambda r: r.get('_tree_score', 0.0), reverse=True)

def build_tree_rag_prompt(query,        # the user question
                          enriched,     # rows from tree_search / tree_expand_results
                          max_ctx=8,    # max context segments
                          ):
    "Assemble a TreeRAG prompt: breadcrumb + section context + content per result."
    parts = []
    for r in enriched[:max_ctx]:
        seg = []
        if r.get('breadcrumb'): seg.append(f"[{r['breadcrumb']}]")
        if r.get('parent_intro'): seg.append(f"Section context: {r['parent_intro']}")
        seg.append(r.get('content', ''))
        parts.append('\n'.join(seg).strip())
    ctx = '\n\n---\n\n'.join(p for p in parts if p)
    return (f'Use the following document sections to answer the question.\n\n'
            f'{ctx}\n\nQuestion: {query}\nAnswer:')

In [ ]:
# end-to-end check: ingest the test PDF, search, and expand with graph context.
import numpy as np, hashlib
from litesearch.core import database

class _HashEncoder:
    "Deterministic hashing encoder - no model download, for tests/examples."
    def __init__(self, dim=64, dtype=np.float16): self.dim, self.dtype = dim, dtype
    def _v(self, t):
        v = np.zeros(self.dim, np.float32)
        for w in re.findall(r'\w+', t.lower()):
            v[int(hashlib.md5(w.encode()).hexdigest(), 16) % self.dim] += 1.0
        n = np.linalg.norm(v)
        return (v / n if n else v).astype(self.dtype)
    def encode_document(self, lns):
        return (np.stack([self._v(t) for t in lns]) if len(lns)
                else np.zeros((0, self.dim), self.dtype))
    def encode_query(self, lns): return self.encode_document(lns)

_db = database(':memory:')
_enc = _HashEncoder()
_res = ingest_pdf_tree(_db, PDF, _enc, chunk_fn=_para_chunks)
assert _res['backend'] == 'oxide', _res['backend']
assert _res['n_nodes'] > 5 and _res['n_edges'] > 0, _res
assert _heading_quality(extract_headings(_doc), _doc) > 0.4, 'clean PDF scored low'

_base = tree_search(_db, 'scaled dot product attention', _enc, expand=False)
_exp = tree_search(_db, 'scaled dot product attention', _enc, expand=True, hops=1)
assert _base, 'no search results'
assert len(_exp) >= len(_base), 'expansion dropped results'
assert any('breadcrumb' in r for r in _exp), 'no breadcrumb attached'

try:
    import docling  # noqa: F401
    print('docling installed - docling headings:', len(docling_headings(PDF)))
except ImportError:
    print('docling not installed - auto-fallback stays on pdf_oxide (expected)')

_prompt = build_tree_rag_prompt('scaled dot product attention', _exp)
assert 'Question:' in _prompt and len(_prompt) > 50, 'bad prompt'
print(f"ingest {_res['n_nodes']} nodes / {_res['n_edges']} edges; "
      f"search {len(_base)} -> {len(_exp)} expanded; top: {_exp[0].get('breadcrumb')}")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()